# Model Egitimi — GitHub Python Projeleri

**Iki tahmin problemi:**
- RQ1-RQ2: `label` — Dosya tekrar degistirilecek mi? (commit tahmini)
- RQ3: `has_bug` — Dosyada hata var mi? (hata tahmini)

**Modeller (3 kategori, 9 model):**
- Traditional ML: Logistic Regression, Random Forest, SVM, XGBoost, LightGBM
- AutoML: AutoGluon WeightedEnsemble
- Deep Learning: MLP, CNN (1D), LSTM

**Feature Setleri:**
- `FEATURES_STATIC` (22): Sadece radon metrikleri
- `FEATURES_STATIC_DERIVED` (26): Radon + turetilmis metrikler
- `FEATURES_ALL` (37): Radon + turetilmis + surec metrikleri + proje bilgileri

**Icerik:**
1. Veri yukleme ve temizlik
2. Sensitivity Analysis (min/max dosya esikleri — her iki hedef icin ayri)
3. Traditional ML egitimi (iki feature seti)
4. Deep Learning egitimi (MLP, CNN, LSTM, iki feature seti)
5. AutoGluon egitimi
6. Toplu sonuc tablolari (makale formatinda)
7. Feature Importance
8. Feature Seti Karsilastirmasi
9. Sonuclari Kaydet

In [ ]:
!pip install pandas scikit-learn xgboost lightgbm autogluon matplotlib seaborn tensorflow imbalanced-learn ipywidgets h2o tpot polars pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
# ============================================================
# AYARLAR
# ============================================================

DATA_DIR = Path(r"C:\Users\bm123\Documents\MetricHunter\Final\output")

# Veri seti dosyasini bul (en son olusturulan full dataset)
full_files = sorted(DATA_DIR.glob("dataset_model_filtered_*.csv"))
if not full_files:
    # Fallback: filtrelenmemis dataset
    full_files = sorted(DATA_DIR.glob("dataset_full_*.csv"))
    if not full_files:
        raise FileNotFoundError(f"{DATA_DIR} icinde dataset bulunamadi!")
    print("UYARI: Filtrelenmis dataset bulunamadi, ham dataset kullaniliyor")
DATA_PATH = full_files[-1]

# Statik metrikler (radon, 22 adet)
STATIC_METRIC_COLS = [
    "loc", "lloc", "sloc", "comments", "multi", "blank", "single_comments",
    "cc_mean", "cc_max", "cc_total", "num_functions",
    "h_vocabulary", "h_length", "h_volume", "h_difficulty",
    "h_effort", "h_bugs", "h_time", "h_calculated_length",
    "maintainability_index", "comment_ratio", "doc_ratio"
]

# Surec metrikleri (git log, 8 adet)
PROCESS_METRIC_COLS = [
    "commit_count", "bug_count", "n_authors", "file_age_days",
    "churn_total", "avg_churn_per_commit", "max_single_churn", "recent_commits_90d"
]

# Turetilmis metrikler (4 adet)
DERIVED_METRIC_COLS = [
    "complexity_density", "comment_per_function", "avg_function_length", "effort_per_line"
]

# Proje bilgileri
PROJECT_COLS = ["stars", "contributor_count", "project_age_days"]

# Feature setleri (deneylerde kullanilacak)
FEATURES_STATIC = STATIC_METRIC_COLS
FEATURES_STATIC_DERIVED = STATIC_METRIC_COLS + DERIVED_METRIC_COLS
FEATURES_ALL = STATIC_METRIC_COLS + DERIVED_METRIC_COLS + PROCESS_METRIC_COLS + PROJECT_COLS

# Label icin: surec metrikleri TAMAMEN cikarilmali (hepsi commit gecmisinden)
FEATURES_ALL_LABEL = STATIC_METRIC_COLS + DERIVED_METRIC_COLS + PROJECT_COLS
# Yani: 22 statik + 4 turetilmis + 3 proje = 29 feature

# Bug icin: commit_count kalabilir, bug_count cikarilir (mevcut hali dogru)
FEATURES_ALL_BUG = [c for c in FEATURES_ALL if c != "bug_count"]

# GroupKFold
N_SPLITS = 5
RANDOM_STATE = 42

# Deep Learning
DL_EPOCHS = 50
DL_BATCH_SIZE = 64

print(f"Dataset: {DATA_PATH.name}")
print(f"FEATURES_STATIC       : {len(FEATURES_STATIC)} feature")
print(f"FEATURES_STATIC_DERIVED: {len(FEATURES_STATIC_DERIVED)} feature")
print(f"FEATURES_ALL_LABEL    : {len(FEATURES_ALL_LABEL)} feature (commit_count cikarildi)")
print(f"FEATURES_ALL_BUG      : {len(FEATURES_ALL_BUG)} feature (bug_count cikarildi)")

---
## 1. Veri Yukleme ve Temizlik

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Ham veri: {len(df_raw)} dosya, {df_raw['project'].nunique()} proje")

# --- Temizlik ---
df = df_raw.copy()

# 1. MI = 0 (radon hesaplayamadi)
mi_zero = (df["maintainability_index"] == 0).sum()
df = df[df["maintainability_index"] > 0]
print(f"MI=0 elendi: {mi_zero} dosya")

# 2. NaN temizle (sadece statik metrikler; surec metrikleri NaN olmaz)
before = len(df)
df = df.dropna(subset=STATIC_METRIC_COLS)
print(f"NaN elendi: {before - len(df)} dosya")

print(f"\nTemiz veri: {len(df)} dosya, {df['project'].nunique()} proje")
print(f"Label dagilimi: 0={(df['label']==0).sum()} ({(df['label']==0).mean():.1%}), 1={(df['label']==1).sum()} ({(df['label']==1).mean():.1%})")
print(f"Bug dagilimi: 0={(df['has_bug']==0).sum()} ({(df['has_bug']==0).mean():.1%}), 1={(df['has_bug']==1).sum()} ({(df['has_bug']==1).mean():.1%})")

In [ ]:
# Proje basina dosya dagilimi
pp = df.groupby("project").size()
print("Proje basina dosya dagilimi:")
print(pp.describe())
print(f"\nPersentiller:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p}: {pp.quantile(p/100):.0f} dosya")

---
## 2. Sensitivity Analysis

Proje basina minimum ve maksimum dosya sayisi esiklerinin model performansina etkisini sistematik olarak analiz ediyoruz.
Her kombinasyon icin GroupKFold(5) ile Random Forest egitip ortalama F1 skorunu kaydediyoruz.

**Neden Random Forest?** Hizli egitilir, hyperparameter'a duyarsiz, sensitivity analysis icin ideal baseline.

In [ ]:
def evaluate_threshold(df_clean, feature_cols, target_col, min_files, max_files, n_splits=5):
    """
    Verilen min/max esiklerle filtreleyip GroupKFold ile F1 hesapla.
    """
    temp = df_clean.copy()
    
    # Min filtresi: kucuk projeleri ele
    project_sizes = temp.groupby("project").size()
    keep = project_sizes[project_sizes >= min_files].index
    temp = temp[temp["project"].isin(keep)]
    
    # Max filtresi: buyuk projelerden ornekle
    # group_keys=False: pandas 2.0+ MultiIndex sorununu onler
    if max_files is not None:
        temp = temp.groupby("project", group_keys=False).apply(
            lambda x: x.sample(min(len(x), max_files), random_state=RANDOM_STATE)
        ).reset_index(drop=True)
    
    n_projects = temp["project"].nunique()
    n_files = len(temp)
    
    # Yeterli veri/proje yoksa atla
    if n_projects < n_splits * 2 or n_files < 100:
        return None, n_projects, n_files
    
    X = temp[feature_cols].values
    y = temp[target_col].values
    groups = temp["project"].values
    
    gkf = GroupKFold(n_splits=n_splits)
    f1_scores = []
    
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        model = RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        f1_scores.append(f1_score(y_test, y_pred, average='weighted'))
    
    return np.mean(f1_scores), n_projects, n_files

In [ ]:
# Sensitivity Analysis: hem label (commit) hem has_bug (hata) icin
min_values  = [1, 3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50]
max_values  = [None, 20, 30, 40, 50, 60, 75, 80, 100, 120, 150, 200, 300]
max_labels  = ["Yok", "20", "30", "40", "50", "60", "75", "80", "100", "120", "150", "200", "300"]

results_label = []
results_bug   = []
total = len(min_values) * len(max_values)

print(f"Sensitivity Analysis basliyor (label + has_bug)...")
print(f"min degerleri  : {min_values}")
print(f"max degerleri  : {max_labels}")
print(f"Toplam kombinasyon: {total} x 2 hedef = {total * 2} RF egitimi\n")

for i, min_f in enumerate(min_values, 1):
    for j, (max_f, max_label) in enumerate(zip(max_values, max_labels), 1):

        # gecersiz kombinasyonu atla: max_f tanimli ve min_f >= max_f
        if max_f is not None and min_f >= max_f:
            skip_msg = "ATLANDI (min>=max)"
            results_label.append({"min_files": min_f, "max_files": max_label,
                                   "f1_score": None, "n_projects": 0, "n_files": 0})
            results_bug.append(  {"min_files": min_f, "max_files": max_label,
                                   "f1_score": None, "n_projects": 0, "n_files": 0})
            print(f"  [{i:>2d}/{len(min_values)}] min={min_f:>3d}, max={max_label:>4s} -> {skip_msg}")
            continue

        f1_l, n_proj_l, n_files_l = evaluate_threshold(df, FEATURES_STATIC, "label",   min_f, max_f)
        f1_b, n_proj_b, n_files_b = evaluate_threshold(df, FEATURES_STATIC, "has_bug", min_f, max_f)

        results_label.append({"min_files": min_f, "max_files": max_label,
                               "f1_score": f1_l, "n_projects": n_proj_l, "n_files": n_files_l})
        results_bug.append(  {"min_files": min_f, "max_files": max_label,
                               "f1_score": f1_b, "n_projects": n_proj_b, "n_files": n_files_b})

        sl = f"{f1_l:.4f}" if f1_l is not None else "YETERSIZ"
        sb = f"{f1_b:.4f}" if f1_b is not None else "YETERSIZ"
        print(f"  [{i:>2d}/{len(min_values)}] min={min_f:>3d}, max={max_label:>4s} "
              f"-> label={sl}, bug={sb} | {n_proj_l} proje, {n_files_l} dosya")

df_sensitivity_label = pd.DataFrame(results_label)
df_sensitivity_bug   = pd.DataFrame(results_bug)

# En iyi sonuclari ozet olarak goster
for name, df_s in [("LABEL", df_sensitivity_label), ("HAS_BUG", df_sensitivity_bug)]:
    valid = df_s.dropna(subset=["f1_score"])
    if not valid.empty:
        best = valid.loc[valid["f1_score"].idxmax()]
        print(f"\n[{name}] En iyi F1: {best['f1_score']:.4f} "
              f"| min={best['min_files']}, max={best['max_files']} "
              f"| {best['n_projects']} proje, {best['n_files']} dosya")

print("\nTamamlandi!")

In [ ]:
# Heatmap: iki hedef yan yana
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for ax, (df_sens, title) in zip(axes, [
    (df_sensitivity_label, "Commit Tahmini (label)"),
    (df_sensitivity_bug,   "Hata Tahmini (has_bug)")
]):
    pivot = df_sens.pivot(index="min_files", columns="max_files", values="f1_score")
    pivot = pivot[max_labels]
    sns.heatmap(
        pivot, annot=True, fmt=".4f", cmap="YlOrRd",
        linewidths=0.5, ax=ax, vmin=pivot.min().min() - 0.01
    )
    ax.set_title(f"Sensitivity Analysis: {title}\n(Random Forest, GroupKFold=5)", fontsize=11)
    ax.set_xlabel("Proje basina maksimum dosya")
    ax.set_ylabel("Proje basina minimum dosya")

plt.suptitle("Proje Basina Dosya Esikleri vs F1 Skoru", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / "sensitivity_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# En iyi kombinasyonlar (her hedef icin ayri)
best_label = df_sensitivity_label.dropna(subset=["f1_score"]).sort_values("f1_score", ascending=False).iloc[0]
best_bug   = df_sensitivity_bug.dropna(subset=["f1_score"]).sort_values("f1_score", ascending=False).iloc[0]

print(f"\nEn iyi kombinasyon (label):")
print(f"  min={best_label['min_files']}, max={best_label['max_files']}, F1={best_label['f1_score']:.4f}, "
      f"Proje={best_label['n_projects']}, Dosya={best_label['n_files']}")
print(f"\nEn iyi kombinasyon (has_bug):")
print(f"  min={best_bug['min_files']}, max={best_bug['max_files']}, F1={best_bug['f1_score']:.4f}, "
      f"Proje={best_bug['n_projects']}, Dosya={best_bug['n_files']}")

In [ ]:
# Proje ve dosya sayisi tablosu (her iki hedef)
for df_sens, tag in [(df_sensitivity_label, "label"), (df_sensitivity_bug, "has_bug")]:
    pivot_proj  = df_sens.pivot(index="min_files", columns="max_files", values="n_projects")
    pivot_files = df_sens.pivot(index="min_files", columns="max_files", values="n_files")
    print(f"[{tag}] Proje sayilari:")
    print(pivot_proj[max_labels].to_string())
    print(f"\n[{tag}] Dosya sayilari:")
    print(pivot_files[max_labels].to_string())
    print()

---
## 3. Optimal Esiklerle Veri Hazirlama

In [ ]:
def apply_thresholds(df_source, best_row, tag):
    """Verilen esikleri uygulayip filtrelenmiş DataFrame dondur."""
    best_min     = int(best_row["min_files"])
    best_max_str = best_row["max_files"]
    best_max     = None if best_max_str == "Yok" else int(best_max_str)

    df_out = df_source.copy()
    project_sizes = df_out.groupby("project").size()
    keep = project_sizes[project_sizes >= best_min].index
    df_out = df_out[df_out["project"].isin(keep)]

    if best_max is not None:
        # group_keys=False: pandas 2.0+ MultiIndex sorununu onler
        df_out = df_out.groupby("project", group_keys=False).apply(
            lambda x: x.sample(min(len(x), best_max), random_state=RANDOM_STATE)
        ).reset_index(drop=True)

    print(f"[{tag}] min={best_min}, max={best_max_str} "
          f"-> {len(df_out)} dosya, {df_out['project'].nunique()} proje")
    print(f"  label : 0={(df_out['label']==0).sum()} ({(df_out['label']==0).mean():.1%}), "
          f"1={(df_out['label']==1).sum()} ({(df_out['label']==1).mean():.1%})")
    print(f"  has_bug: 0={(df_out['has_bug']==0).sum()} ({(df_out['has_bug']==0).mean():.1%}), "
          f"1={(df_out['has_bug']==1).sum()} ({(df_out['has_bug']==1).mean():.1%})")
    return df_out, best_min, best_max_str

print("Optimal esikler uygulanıyor...\n")
df_model_label, BEST_MIN_LABEL, BEST_MAX_LABEL = apply_thresholds(df, best_label, "label")
print()
df_model_bug,   BEST_MIN_BUG,   BEST_MAX_BUG   = apply_thresholds(df, best_bug,   "has_bug")

---
## 4. Traditional ML Modelleri

Her model GroupKFold(5) ile degerlendirilir. Her fold'da farkli projeler test'te olur.

In [ ]:
def evaluate_sklearn_models(df_data, feature_cols, target_col, models_dict,
                             n_splits=5, use_smote=False):
    """
    Sklearn modellerini GroupKFold ile egitip degerlendir.
    use_smote=True: her fold'un train setine SMOTE uygulanir.
                    SMOTE sadece train'e dokunur, test seti temiz kalir.
    """
    X = df_data[feature_cols].values
    y = df_data[target_col].values
    groups = df_data["project"].values

    gkf = GroupKFold(n_splits=n_splits)
    all_results = []

    for model_name, model_fn in models_dict.items():
        print(f"  {model_name} egitiliyor...", end=" ")

        fold_metrics = []

        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s  = scaler.transform(X_test)

            # SMOTE: sadece train setine uygula, test setine dokunma
            if use_smote:
                smote = SMOTE(random_state=RANDOM_STATE)
                X_train_s, y_train = smote.fit_resample(X_train_s, y_train)

            model = model_fn()
            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_test_s)

            try:
                y_proba = model.predict_proba(X_test_s)[:, 1]
                roc = roc_auc_score(y_test, y_proba)
            except Exception:
                roc = None

            fold_metrics.append({
                "accuracy":     accuracy_score(y_test, y_pred),
                "precision":    precision_score(y_test, y_pred, average="weighted", zero_division=0),
                "recall":       recall_score(y_test, y_pred, average="weighted", zero_division=0),
                "f1":           f1_score(y_test, y_pred, average="weighted"),
                "balanced_acc": balanced_accuracy_score(y_test, y_pred),
                "mcc":          matthews_corrcoef(y_test, y_pred),
                "roc_auc":      roc
            })

        avg = {}
        for key in fold_metrics[0]:
            vals = [fm[key] for fm in fold_metrics if fm[key] is not None]
            avg[key] = np.mean(vals) if vals else None

        avg["model"]    = model_name
        avg["category"] = "Traditional ML"
        all_results.append(avg)
        print(f"F1={avg['f1']:.4f}, Acc={avg['accuracy']:.4f}")

    return all_results

In [ ]:
# ============================================================
# Model tanimlari — class balancing YOK (has_bug artik dengeli)
# ============================================================
traditional_models = {
    "Logistic Regression": lambda: LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=200, random_state=RANDOM_STATE,
        eval_metric="logloss", verbosity=0
    ),
    "LightGBM": lambda: LGBMClassifier(
        n_estimators=200, random_state=RANDOM_STATE, verbose=-1
    ),
    "SVM (RBF)": lambda: SVC(
        kernel="rbf", probability=True, random_state=RANDOM_STATE,
        cache_size=500, max_iter=10000
    ),
}

print("Model tanimlari hazirlandi.")

In [ ]:
# --- RQ1-RQ2: Commit Tahmini (label) ---
print("=" * 60)
print("TRADITIONAL ML: COMMIT TAHMINI — Sadece Statik Metrikler")
print("=" * 60)
trad_label_static = evaluate_sklearn_models(df_model_label, FEATURES_STATIC, "label", traditional_models)

print("\n" + "=" * 60)
print("TRADITIONAL ML: COMMIT TAHMINI — Tum Feature'lar")
print("=" * 60)
trad_label_all = evaluate_sklearn_models(df_model_label, FEATURES_ALL_LABEL, "label", traditional_models)

In [ ]:
# --- RQ3: Hata Tahmini (has_bug) ---
print("=" * 60)
print("TRADITIONAL ML: HATA TAHMINI — Sadece Statik Metrikler")
print("=" * 60)
trad_bug_static = evaluate_sklearn_models(df_model_bug, FEATURES_STATIC, "has_bug", traditional_models)

print("\n" + "=" * 60)
print("TRADITIONAL ML: HATA TAHMINI — Tum Feature'lar")
print("=" * 60)
trad_bug_all = evaluate_sklearn_models(df_model_bug, FEATURES_ALL_BUG, "has_bug", traditional_models)

---
## 5. Deep Learning Modelleri (MLP, CNN, LSTM)

- **MLP**: Tam bagli katmanlar, metrik vektorunu dogrudan isler
- **CNN (1D)**: Metrik vektorune konvolusyon filtresi uygular, yerel oruntuleri yakalar
- **LSTM**: Metrik vektorunu sirali veri olarak isler (statik metriklerde sirali yapi yok, dusuk performans beklenir)

In [ ]:
def build_mlp(input_dim):
    """Multi-Layer Perceptron: tam bagli katmanlar."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_cnn(input_dim):
    """1D CNN: metrik vektorune konvolusyon filtresi uygular."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Reshape((input_dim, 1)),
        tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        tf.keras.layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_lstm(input_dim):
    """LSTM: metrik vektorunu sirali veri olarak isler."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Reshape((input_dim, 1)),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
def evaluate_dl_models(df_data, feature_cols, target_col, n_splits=5, use_class_weight=False):
    """
    Deep Learning modellerini GroupKFold ile egitip degerlendir.
    use_class_weight=True: her fold'da train setinden compute_class_weight ile agirliklar
                           hesaplanip model.fit(..., class_weight=...) olarak gecilir.
    """
    X = df_data[feature_cols].values
    y = df_data[target_col].values
    groups = df_data["project"].values
    input_dim = X.shape[1]

    dl_models = {
        "MLP":      build_mlp,
        "CNN (1D)": build_cnn,
        "LSTM":     build_lstm,
    }

    gkf = GroupKFold(n_splits=n_splits)
    all_results = []

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    for model_name, build_fn in dl_models.items():
        print(f"  {model_name} egitiliyor...", end=" ")

        fold_metrics = []

        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s  = scaler.transform(X_test)

            # class_weight: her fold'un train setinden ayri hesapla
            cw = None
            if use_class_weight:
                classes = np.unique(y_train)
                weights = compute_class_weight("balanced", classes=classes, y=y_train)
                cw = dict(zip(classes, weights.tolist()))

            # Her fold icin yeni model
            tf.keras.backend.clear_session()
            model = build_fn(input_dim)

            model.fit(
                X_train_s, y_train,
                epochs=DL_EPOCHS,
                batch_size=DL_BATCH_SIZE,
                validation_split=0.15,
                callbacks=[early_stop],
                class_weight=cw,
                verbose=0
            )

            y_proba = model.predict(X_test_s, verbose=0).flatten()
            y_pred  = (y_proba >= 0.5).astype(int)

            try:
                roc = roc_auc_score(y_test, y_proba)
            except Exception:
                roc = None

            fold_metrics.append({
                "accuracy":     accuracy_score(y_test, y_pred),
                "precision":    precision_score(y_test, y_pred, average="weighted", zero_division=0),
                "recall":       recall_score(y_test, y_pred, average="weighted", zero_division=0),
                "f1":           f1_score(y_test, y_pred, average="weighted"),
                "balanced_acc": balanced_accuracy_score(y_test, y_pred),
                "mcc":          matthews_corrcoef(y_test, y_pred),
                "roc_auc":      roc
            })

        avg = {}
        for key in fold_metrics[0]:
            vals = [fm[key] for fm in fold_metrics if fm[key] is not None]
            avg[key] = np.mean(vals) if vals else None

        avg["model"]    = model_name
        avg["category"] = "Deep Learning"
        all_results.append(avg)
        print(f"F1={avg['f1']:.4f}, Acc={avg['accuracy']:.4f}")

    return all_results

In [ ]:
# --- RQ1-RQ2: Commit Tahmini (label) ---
print("=" * 60)
print("DEEP LEARNING: COMMIT TAHMINI — Sadece Statik Metrikler")
print("=" * 60)
dl_label_static = evaluate_dl_models(df_model_label, FEATURES_STATIC, "label")

print("\n" + "=" * 60)
print("DEEP LEARNING: COMMIT TAHMINI — Tum Feature'lar")
print("=" * 60)
dl_label_all = evaluate_dl_models(df_model_label, FEATURES_ALL_LABEL, "label")

In [ ]:
# --- RQ3: Hata Tahmini (has_bug) ---
print("=" * 60)
print("DEEP LEARNING: HATA TAHMINI — Sadece Statik Metrikler")
print("=" * 60)
dl_bug_static = evaluate_dl_models(df_model_bug, FEATURES_STATIC, "has_bug")

print("\n" + "=" * 60)
print("DEEP LEARNING: HATA TAHMINI — Tum Feature'lar")
print("=" * 60)
dl_bug_all = evaluate_dl_models(df_model_bug, FEATURES_ALL_BUG, "has_bug")

---
## 6. AutoGluon

In [ ]:
from autogluon.tabular import TabularPredictor
import shutil

def evaluate_autogluon(df_data, feature_cols, target_col, n_splits=5, time_limit=600):
    """
    AutoGluon'u GroupKFold ile degerlendir.
    time_limit: fold basina saniye (varsayilan 600s).
    dynamic_stacking=False: DyStack on islemine sure harcamayi onler,
    time_limit asiminin onune gecer.
    """
    X = df_data[feature_cols + [target_col, "project"]].copy()
    groups = X["project"]

    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, X[target_col], groups)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        train_data = X.iloc[train_idx][feature_cols + [target_col]]
        test_data  = X.iloc[test_idx][feature_cols + [target_col]]

        predictor = TabularPredictor(
            label=target_col,
            eval_metric="f1_weighted",
            verbosity=0
        ).fit(
            train_data,
            time_limit=time_limit,
            presets="good_quality",
            dynamic_stacking=False   # DyStack on-islem suresini atlayarak time_limit asimini onler
        )

        y_test = test_data[target_col].values
        y_pred = predictor.predict(test_data.drop(columns=[target_col])).values

        try:
            y_proba = predictor.predict_proba(test_data.drop(columns=[target_col]))
            if isinstance(y_proba, pd.DataFrame):
                y_proba = y_proba[1].values
            roc = roc_auc_score(y_test, y_proba)
        except Exception:
            roc = None

        f1  = f1_score(y_test, y_pred, average="weighted")
        acc = accuracy_score(y_test, y_pred)

        fold_metrics.append({
            "accuracy":     acc,
            "precision":    precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall":       recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1":           f1,
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "mcc":          matthews_corrcoef(y_test, y_pred),
            "roc_auc":      roc
        })
        print(f"F1={f1:.4f}, Acc={acc:.4f}")

        # Gecici dosyalari temizle
        if hasattr(predictor, 'path'):
            shutil.rmtree(predictor.path, ignore_errors=True)

    avg = {}
    for key in fold_metrics[0]:
        vals = [fm[key] for fm in fold_metrics if fm[key] is not None]
        avg[key] = np.mean(vals) if vals else None
    avg["model"]    = "AutoGluon"
    avg["category"] = "AutoML"
    return [avg]

In [ ]:
# --- RQ1-RQ2: Commit Tahmini ---
print("=" * 60)
print("AUTOGLUON: COMMIT TAHMINI (label)")
print("=" * 60)
ag_label = evaluate_autogluon(df_model_label, FEATURES_ALL_LABEL, "label", time_limit=900)

In [ ]:
# --- RQ3: Hata Tahmini ---
print("=" * 60)
print("AUTOGLUON: HATA TAHMINI (has_bug)")
print("=" * 60)
ag_bug = evaluate_autogluon(df_model_bug, FEATURES_ALL_BUG, "has_bug", time_limit=900)

---
## 6b. H2O AutoML

H2O, JVM tabanlı dağıtık bir AutoML framework'üdür. GBM, XGBoost, Random Forest,
Deep Learning ve Stacked Ensembles'ı otomatik olarak dener ve ağırlıklı ensemble kurar.
`max_runtime_secs` ile fold başına süre sınırlanır.

In [ ]:
import h2o
from h2o.automl import H2OAutoML

def evaluate_h2o(df_data, feature_cols, target_col, n_splits=5, max_runtime_secs=120):
    """
    H2O AutoML'i GroupKFold ile degerlendir.
    Her fold sonunda h2o.remove_all() ile bellek temizlenir.
    Son fold sonunda h2o.shutdown() ile JVM kapatilir.
    """
    h2o.init(nthreads=-1, max_mem_size="4G", verbose=False)

    X = df_data[feature_cols + [target_col, "project"]].copy()
    groups = X["project"]

    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, X[target_col], groups)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        train_data = X.iloc[train_idx][feature_cols + [target_col]]
        test_data  = X.iloc[test_idx][feature_cols + [target_col]]

        # pandas -> h2o frame
        h_train = h2o.H2OFrame(train_data)
        h_test  = h2o.H2OFrame(test_data)

        # target'i factor yap (classification icin)
        h_train[target_col] = h_train[target_col].asfactor()
        h_test[target_col]  = h_test[target_col].asfactor()

        aml = H2OAutoML(
            max_runtime_secs=max_runtime_secs,
            max_models=20,
            seed=RANDOM_STATE,
            sort_metric="AUTO",
            verbosity="warn"
        )
        aml.train(x=feature_cols, y=target_col, training_frame=h_train)

        # En iyi modelin tahmini
        preds  = aml.leader.predict(h_test).as_data_frame()
        y_test = test_data[target_col].values
        y_pred = preds["predict"].astype(int).values

        try:
            y_proba = preds["p1"].values
            roc = roc_auc_score(y_test, y_proba)
        except Exception:
            roc = None

        f1 = f1_score(y_test, y_pred, average="weighted")
        fold_metrics.append({
            "accuracy":     accuracy_score(y_test, y_pred),
            "precision":    precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall":       recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1":           f1,
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "mcc":          matthews_corrcoef(y_test, y_pred),
            "roc_auc":      roc
        })
        print(f"F1={f1:.4f}")

        h2o.remove_all()  # fold bellegini temizle

    avg = {}
    for key in fold_metrics[0]:
        vals = [fm[key] for fm in fold_metrics if fm[key] is not None]
        avg[key] = np.mean(vals) if vals else None
    avg["model"]    = "H2O AutoML"
    avg["category"] = "AutoML"

    h2o.shutdown(prompt=False)
    return [avg]

In [ ]:
print("=" * 60)
print("H2O AutoML: COMMIT TAHMINI (label)")
print("=" * 60)
h2o_label = evaluate_h2o(df_model_label, FEATURES_ALL_LABEL, "label", max_runtime_secs=600)
print(f"H2O (label): F1={h2o_label[0]['f1']:.4f}")

In [ ]:
print("=" * 60)
print("H2O AutoML: HATA TAHMINI (has_bug)")
print("=" * 60)
h2o_bug = evaluate_h2o(df_model_bug, FEATURES_ALL_BUG, "has_bug", max_runtime_secs=600)
print(f"H2O (has_bug): F1={h2o_bug[0]['f1']:.4f}")

---
## 6c. TPOT

TPOT, genetik programlama ile en iyi scikit-learn pipeline'ını arar.
Her generasyonda preprocessing + model kombinasyonlarını evrimleştirir.
`max_time_mins` ile fold başına süre, `max_eval_time_mins` ile
tek pipeline değerlendirme süresi sınırlanır.

In [ ]:
from tpot import TPOTClassifier

def evaluate_tpot(df_data, feature_cols, target_col, n_splits=5, max_time_mins=5):
    """
    TPOT'u GroupKFold ile degerlendir.
    TPOT genetik algoritma ile en iyi scikit-learn pipeline'ini arar.
    """
    X = df_data[feature_cols].values
    y = df_data[target_col].values
    groups = df_data["project"].values

    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)
        
        tpot = TPOTClassifier(
            search_space="linear-light",
            scorers=["f1_weighted"],
            scorers_weights=[1],
            max_time_mins=max_time_mins,
            max_eval_time_mins=1,
            n_jobs=1,
            verbose=0,
            random_state=RANDOM_STATE,
        )
        tpot.fit(X_train_s, y_train)
        y_pred = tpot.predict(X_test_s)

        try:
            y_proba = tpot.predict_proba(X_test_s)[:, 1]
            roc = roc_auc_score(y_test, y_proba)
        except Exception:
            roc = None

        f1 = f1_score(y_test, y_pred, average="weighted")
        fold_metrics.append({
            "accuracy":     accuracy_score(y_test, y_pred),
            "precision":    precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall":       recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1":           f1,
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "mcc":          matthews_corrcoef(y_test, y_pred),
            "roc_auc":      roc
        })
        print(f"F1={f1:.4f}")

    avg = {}
    for key in fold_metrics[0]:
        vals = [fm[key] for fm in fold_metrics if fm[key] is not None]
        avg[key] = np.mean(vals) if vals else None
    avg["model"]    = "TPOT"
    avg["category"] = "AutoML"
    return [avg]

In [ ]:
print("=" * 60)
print("TPOT: COMMIT TAHMINI (label)")
print("=" * 60)
tpot_label = evaluate_tpot(df_model_label, FEATURES_ALL_LABEL, "label", max_time_mins=5)
print(f"TPOT (label): F1={tpot_label[0]['f1']:.4f}")

In [ ]:
print("=" * 60)
print("TPOT: HATA TAHMINI (has_bug)")
print("=" * 60)
tpot_bug = evaluate_tpot(df_model_bug, FEATURES_ALL_BUG, "has_bug", max_time_mins=5)
print(f"TPOT (has_bug): F1={tpot_bug[0]['f1']:.4f}")

---
## 7. Toplu Sonuc Tablolari (Makale Formatinda)

Uc kategori ayri tablo:
- Table A: Traditional ML (Top 5 by F1)
- Table B: Deep Learning (Top 3 by F1)
- Table C: AutoML (AutoGluon)

In [ ]:
display_cols = ["accuracy", "precision", "recall", "f1", "balanced_acc", "mcc", "roc_auc"]

def format_results(results_list, title):
    """Sonuclari DataFrame olarak formatla ve yazdir."""
    df_r = pd.DataFrame(results_list).set_index("model")
    df_r = df_r.sort_values("f1", ascending=False)
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(df_r[display_cols].round(4).to_string())
    return df_r

In [ ]:
# ========== COMMIT TAHMINI (label) ==========
print("\n" + "#" * 80)
print("#  RQ1-RQ2: COMMIT TAHMINI (label) - TUM SONUCLAR")
print("#" * 80)

print("\nCOMMIT TAHMINI — Sadece Statik Metrikler")
all_label_static = pd.concat([
    pd.DataFrame(trad_label_static).set_index("model"),
    pd.DataFrame(dl_label_static).set_index("model"),
])
print(all_label_static.sort_values("f1", ascending=False)[display_cols].round(4).to_string())

print("\nCOMMIT TAHMINI — Tum Feature'lar (statik + surec + proje)")
all_label_all = pd.concat([
    pd.DataFrame(trad_label_all).set_index("model"),
    pd.DataFrame(dl_label_all).set_index("model"),
    pd.DataFrame(ag_label).set_index("model"),
    pd.DataFrame(h2o_label).set_index("model"),
    pd.DataFrame(tpot_label).set_index("model"),
])
print(all_label_all.sort_values("f1", ascending=False)[display_cols].round(4).to_string())

In [ ]:
# ========== HATA TAHMINI (has_bug) ==========
print("\n" + "#" * 80)
print("#  RQ3: HATA TAHMINI (has_bug) - TUM SONUCLAR")
print("#" * 80)

print("\nHATA TAHMINI — Sadece Statik Metrikler")
all_bug_static = pd.concat([
    pd.DataFrame(trad_bug_static).set_index("model"),
    pd.DataFrame(dl_bug_static).set_index("model"),
])
print(all_bug_static.sort_values("f1", ascending=False)[display_cols].round(4).to_string())

print("\nHATA TAHMINI — Tum Feature'lar (statik + surec + proje)")
all_bug_all = pd.concat([
    pd.DataFrame(trad_bug_all).set_index("model"),
    pd.DataFrame(dl_bug_all).set_index("model"),
    pd.DataFrame(ag_bug).set_index("model"),
    pd.DataFrame(h2o_bug).set_index("model"),
    pd.DataFrame(tpot_bug).set_index("model"),
])
print(all_bug_all.sort_values("f1", ascending=False)[display_cols].round(4).to_string())

---
## 6d. Stacking Hibrit Model

 RF + XGBoost + LightGBM — her biri kendi tahminini üretir  
Logistic Regression — base tahminleri birleştirerek final kararı verir

Stacking içindeki `cv=5` kendi cross-validation'ını yapıyor; bu GroupKFold'un dışındaki
bir iç CV — grup yapısını bilmiyor. Dış GroupKFold proje sızıntısını engelliyor.

In [ ]:
def evaluate_stacking_hybrid(df_data, feature_cols, target_col,
                              all_results_df, n_splits=5):
    """
    Onceki sonuclara bakarak en iyi Traditional ML + en iyi AutoML'i secip
    manual stacking ile birlestirir.
    
    Base modeller onceki sonuclardan otomatik secilir.
    Meta model: Logistic Regression.
    AutoML modelleri sklearn-uyumlu olmadigindan manual stacking yapilir.
    """
    # --- Adim 1: Onceki sonuclardan en iyi modelleri sec ---
    trad_models = all_results_df[all_results_df["category"] == "Traditional ML"]
    automl_models = all_results_df[all_results_df["category"] == "AutoML"]
    
    best_trad_name = trad_models["f1"].idxmax()
    best_automl_name = automl_models["f1"].idxmax()
    
    print(f"  Secilen modeller:")
    print(f"    Traditional ML: {best_trad_name} (F1={trad_models.loc[best_trad_name, 'f1']:.4f})")
    print(f"    AutoML:         {best_automl_name} (F1={automl_models.loc[best_automl_name, 'f1']:.4f})")
    
    # --- Traditional ML model fabrikasi ---
    trad_factory = {
        "Logistic Regression": lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Random Forest": lambda: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
        "XGBoost": lambda: XGBClassifier(n_estimators=200, random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0),
        "LightGBM": lambda: LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, verbose=-1),
        "SVM (RBF)": lambda: SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE, cache_size=500, max_iter=10000),
    }
    
    X = df_data[feature_cols].values
    y = df_data[target_col].values
    groups = df_data["project"].values
    
    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []
    
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        
        # --- Base Model 1: Traditional ML ---
        trad_model = trad_factory[best_trad_name]()
        trad_model.fit(X_train_s, y_train)
        trad_proba_test = trad_model.predict_proba(X_train_s)[:, 1]  # meta icin train
        trad_proba_eval = trad_model.predict_proba(X_test_s)[:, 1]   # final test
        
        # --- Base Model 2: AutoML ---
        # AutoML modelleri icin train/meta split gerekiyor
        # Train verisinin %80'ini base egitim, %20'sini meta egitim icin ayir
        from sklearn.model_selection import train_test_split as tts
        meta_split_idx = int(len(X_train_s) * 0.8)
        X_base_train = X_train_s[:meta_split_idx]
        y_base_train = y_train[:meta_split_idx]
        X_meta_train = X_train_s[meta_split_idx:]
        y_meta_train = y_train[meta_split_idx:]
        
        # Traditional ML'i base_train ile yeniden egit (meta icin)
        trad_model_meta = trad_factory[best_trad_name]()
        trad_model_meta.fit(X_base_train, y_base_train)
        trad_meta_proba = trad_model_meta.predict_proba(X_meta_train)[:, 1]
        
        automl_meta_proba = None
        automl_eval_proba = None
        
        if best_automl_name == "AutoGluon":
            from autogluon.tabular import TabularPredictor
            import shutil
            
            # Base train
            ag_train = pd.DataFrame(X_base_train, columns=feature_cols)
            ag_train[target_col] = y_base_train
            
            predictor = TabularPredictor(
                label=target_col, eval_metric="f1_weighted", verbosity=0
            ).fit(ag_train, time_limit=120, presets="good_quality", dynamic_stacking=False)
            
            # Meta train olasiliklari
            ag_meta_input = pd.DataFrame(X_meta_train, columns=feature_cols)
            ag_meta_pred = predictor.predict_proba(ag_meta_input)
            automl_meta_proba = ag_meta_pred[1].values if isinstance(ag_meta_pred, pd.DataFrame) else ag_meta_pred[:, 1]
            
            # Test olasiliklari
            ag_test_input = pd.DataFrame(X_test_s, columns=feature_cols)
            ag_test_pred = predictor.predict_proba(ag_test_input)
            automl_eval_proba = ag_test_pred[1].values if isinstance(ag_test_pred, pd.DataFrame) else ag_test_pred[:, 1]
            
            if hasattr(predictor, 'path'):
                shutil.rmtree(predictor.path, ignore_errors=True)
        
        elif best_automl_name == "H2O AutoML":
            import h2o
            from h2o.automl import H2OAutoML
            
            h2o.init(nthreads=-1, max_mem_size="4G", verbose=False)
            
            # Base train
            h_base = h2o.H2OFrame(
                pd.DataFrame(X_base_train, columns=feature_cols).assign(**{target_col: y_base_train})
            )
            h_base[target_col] = h_base[target_col].asfactor()
            
            aml = H2OAutoML(max_runtime_secs=120, max_models=20, seed=RANDOM_STATE, verbosity="warn")
            aml.train(x=feature_cols, y=target_col, training_frame=h_base)
            
            # Meta train
            h_meta = h2o.H2OFrame(pd.DataFrame(X_meta_train, columns=feature_cols))
            meta_preds = aml.leader.predict(h_meta).as_data_frame()
            automl_meta_proba = meta_preds["p1"].values
            
            # Test
            h_test = h2o.H2OFrame(pd.DataFrame(X_test_s, columns=feature_cols))
            test_preds = aml.leader.predict(h_test).as_data_frame()
            automl_eval_proba = test_preds["p1"].values
            
            h2o.remove_all()
            h2o.shutdown(prompt=False)
        
        elif best_automl_name == "TPOT":
            from tpot import TPOTClassifier
            
            tpot = TPOTClassifier(
                search_space="linear-light", scorers=["f1_weighted"],
                scorers_weights=[1], max_time_mins=3,
                max_eval_time_mins=1, n_jobs=1, verbose=0,
                random_state=RANDOM_STATE,
            )
            tpot.fit(X_base_train, y_base_train)
            automl_meta_proba = tpot.predict_proba(X_meta_train)[:, 1]
            automl_eval_proba = tpot.predict_proba(X_test_s)[:, 1]
        
        # --- Meta Model: Logistic Regression ---
        meta_features_train = np.column_stack([trad_meta_proba, automl_meta_proba])
        meta_features_test = np.column_stack([trad_proba_eval, automl_eval_proba])
        
        meta_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        meta_model.fit(meta_features_train, y_meta_train)
        
        y_pred = meta_model.predict(meta_features_test)
        
        try:
            y_proba = meta_model.predict_proba(meta_features_test)[:, 1]
            roc = roc_auc_score(y_test, y_proba)
        except Exception:
            roc = None
        
        f1 = f1_score(y_test, y_pred, average="weighted")
        fold_metrics.append({
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1": f1,
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "mcc": matthews_corrcoef(y_test, y_pred),
            "roc_auc": roc
        })
        print(f"F1={f1:.4f}")
    
    avg = {}
    for key in fold_metrics[0]:
        vals = [fm[key] for fm in fold_metrics if fm[key] is not None]
        avg[key] = np.mean(vals) if vals else None
    avg["model"] = f"Stacking Hibrit ({best_trad_name} + {best_automl_name})"
    avg["category"] = "Hibrit"
    return [avg]

In [ ]:
print("=" * 60)
print("STACKING HIBRIT: COMMIT TAHMINI (label)")
print("=" * 60)
stack_label = evaluate_stacking_hybrid(
    df_model_label, FEATURES_ALL_LABEL, "label",
    all_label_all
)
print(f"Stacking (label): F1={stack_label[0]['f1']:.4f}")

In [ ]:
print("=" * 60)
print("STACKING HIBRIT: HATA TAHMINI (has_bug)")
print("=" * 60)
stack_bug = evaluate_stacking_hybrid(
    df_model_bug, FEATURES_ALL_BUG, "has_bug",
    all_bug_all
)
print(f"Stacking (has_bug): F1={stack_bug[0]['f1']:.4f}")

In [ ]:
# Stacking sonuclarini tablolara ekle
all_label_all = pd.concat([all_label_all, pd.DataFrame(stack_label).set_index("model")])
all_label_all = all_label_all.sort_values("f1", ascending=False)

all_bug_all = pd.concat([all_bug_all, pd.DataFrame(stack_bug).set_index("model")])
all_bug_all = all_bug_all.sort_values("f1", ascending=False)

print("FINAL — Commit Tahmini (stacking dahil):")
print(all_label_all[display_cols].round(4).to_string())
print("\nFINAL — Hata Tahmini (stacking dahil):")
print(all_bug_all[display_cols].round(4).to_string())

In [ ]:
# F1 karsilastirma grafigi
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

category_colors = {
    "Traditional ML": "#3498db",
    "Deep Learning":  "#e74c3c",
    "AutoML":         "#2ecc71",
    "Hibrit":         "#9b59b6",
}

for ax, (title, results_df) in zip(axes, [
    ("Commit Tahmini (label)",   all_label_all),
    ("Hata Tahmini (has_bug)",   all_bug_all)
]):
    sorted_df = results_df.sort_values("f1", ascending=True)
    colors = [category_colors.get(cat, "gray") for cat in sorted_df["category"]]

    bars = ax.barh(sorted_df.index, sorted_df["f1"], color=colors, edgecolor="white", linewidth=1.5)

    for bar, val in zip(bars, sorted_df["f1"]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va='center', fontsize=9, fontweight='bold')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("F1 Score (weighted)")
    ax.set_xlim(0, 1.08)

legend_elements = [mpatches.Patch(facecolor=c, label=l) for l, c in category_colors.items()]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=11, bbox_to_anchor=(0.5, 1.02))

plt.suptitle(f"Model Karsilastirmasi (GroupKFold={N_SPLITS})", fontsize=14, fontweight='bold', y=1.06)
plt.tight_layout()
plt.savefig(DATA_DIR / "model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Feature Importance

In [ ]:
# Feature Importance (Random Forest, commit tahmini, df_model_label uzerinde)
X_all = df_model_label[FEATURES_ALL_LABEL].values
y_all = df_model_label["label"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_scaled, y_all)

importance = pd.Series(rf.feature_importances_, index=FEATURES_ALL_LABEL).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
importance.head(10).plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Feature Importance — Top 10 (Random Forest, Commit Tahmini)", fontsize=13, fontweight='bold')
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_DIR / "feature_importance.png", dpi=150)
plt.show()

print(f"En onemli 10 feature:")
for name, val in importance.head(10).items():
    print(f"  {name}: {val:.4f}")

---
## 9. Feature Seti Karsilastirmasi

In [ ]:
print("=" * 60)
print("FEATURE SETI KARSILASTIRMASI")
print("=" * 60)

# RF ile hizli karsilastirma: 4 feature seti
feature_sets = {
    "Sadece Statik (22)": FEATURES_STATIC,
    "Statik + Turetilmis (26)": FEATURES_STATIC_DERIVED,
    "Tum Feature'lar (29)": FEATURES_ALL_LABEL,
}

rf_model = {"Random Forest": lambda: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)}

print("\n--- Commit Tahmini (label) ---")
for name, cols in feature_sets.items():
    result = evaluate_sklearn_models(df_model_label, cols, "label", rf_model)
    print(f"  {name}: F1={result[0]['f1']:.4f}")

# Bug tahmini icin commit_count kalabilir ama bug_count cikarilmali
feature_sets_bug = {
    "Sadece Statik (22)": FEATURES_STATIC,
    "Statik + Turetilmis (26)": FEATURES_STATIC_DERIVED,
    "Statik + Turetilmis + Surec (34)": FEATURES_STATIC_DERIVED + [c for c in PROCESS_METRIC_COLS if c != "bug_count"],
    "Tum Feature'lar (37)": FEATURES_ALL_BUG,
}

print("\n--- Hata Tahmini (has_bug) ---")
for name, cols in feature_sets_bug.items():
    result = evaluate_sklearn_models(df_model_bug, cols, "has_bug", rf_model)
    print(f"  {name}: F1={result[0]['f1']:.4f}")

---
## 10. Sonuclari Kaydet

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

all_label_all.to_csv(DATA_DIR / f"results_label_{timestamp}.csv")
all_bug_all.to_csv(DATA_DIR / f"results_bug_{timestamp}.csv")
df_sensitivity_label.to_csv(DATA_DIR / f"sensitivity_label_{timestamp}.csv", index=False)
df_sensitivity_bug.to_csv(DATA_DIR / f"sensitivity_bug_{timestamp}.csv",     index=False)

print("Kaydedilen dosyalar:")
print(f"  results_label_{timestamp}.csv")
print(f"  results_bug_{timestamp}.csv")
print(f"  sensitivity_label_{timestamp}.csv")
print(f"  sensitivity_bug_{timestamp}.csv")
print(f"  sensitivity_analysis.png")
print(f"  model_comparison.png")
print(f"  feature_importance.png")

print(f"\n{'='*60}")
print(f"GENEL OZET")
print(f"{'='*60}")
print(f"Commit verisi : {len(df_model_label)} dosya, {df_model_label['project'].nunique()} proje "
      f"(min={BEST_MIN_LABEL}, max={BEST_MAX_LABEL})")
print(f"Bug verisi    : {len(df_model_bug)} dosya, {df_model_bug['project'].nunique()} proje "
      f"(min={BEST_MIN_BUG}, max={BEST_MAX_BUG})")
print(f"Degerlendirme : GroupKFold (k={N_SPLITS})")
print(f"Statik feature: {len(FEATURES_STATIC)}, Tum feature (label): {len(FEATURES_ALL_LABEL)}, "
      f"Tum feature (bug): {len(FEATURES_ALL_BUG)}")

print(f"\n--- En Iyi Modeller (Tum Feature'lar) ---")
print(f"Commit tahmini: {all_label_all['f1'].idxmax()} "
      f"(F1={all_label_all['f1'].max():.4f})")
print(f"Hata tahmini  : {all_bug_all['f1'].idxmax()} "
      f"(F1={all_bug_all['f1'].max():.4f})")

print(f"\n--- En Iyi Modeller (Sadece Statik) ---")
print(f"Commit tahmini: {all_label_static['f1'].idxmax()} "
      f"(F1={all_label_static['f1'].max():.4f})")
print(f"Hata tahmini  : {all_bug_static['f1'].idxmax()} "
      f"(F1={all_bug_static['f1'].max():.4f})")

print(f"\n--- Kategori Bazli En Iyi ---")
for cat in ["Traditional ML", "Deep Learning", "AutoML", "Hibrit"]:
    sub = all_label_all[all_label_all['category'] == cat]
    if len(sub) > 0:
        print(f"  [{cat}] Commit: {sub['f1'].idxmax()} (F1={sub['f1'].max():.4f})")
    sub_b = all_bug_all[all_bug_all['category'] == cat]
    if len(sub_b) > 0:
        print(f"  [{cat}] Bug   : {sub_b['f1'].idxmax()} (F1={sub_b['f1'].max():.4f})")